# Cravveo Company — 파인튜닝 노트북

**프로젝트**: 100일 AI 파인튜닝 도전기  
**환경**: Google Colab T4 GPU  
**모델**: ~~unsloth/gemma-2b-it~~ → **unsloth/Llama-3.2-1B-Instruct** (Day 026에서 교체)

---

## ⚠️ 매 세션 시작 규칙
1. **셀 1~6을 순서대로 실행**
2. **셀 6 실행 후 반드시 런타임 재시작** (상단 메뉴 → 런타임 → 런타임 재시작)
3. 재시작 후 **셀 6.5 → 셀 7**부터 이어서 실행

---

## 🔧 SECTION 0 — 매 세션 시작마다 실행 (Run Every Session)

In [ ]:
# 셀 1 — GPU 확인
!nvidia-smi

In [ ]:
# 셀 2 — Unsloth + torchvision 설치 (반드시 한 셀에 같이)
%%capture
!pip install unsloth
!pip install --upgrade "torchvision>=0.27.0"

In [ ]:
# 셀 3 — Unsloth 설치 확인
import unsloth
print("Unsloth version:", unsloth.__version__)
print("✅ Unsloth 설치 완료")

In [ ]:
# 셀 4 — 나머지 라이브러리 설치
%%capture
!pip install transformers datasets accelerate peft trl bitsandbytes

In [ ]:
# 셀 5 — huggingface_hub 설치
%%capture
!pip install -q huggingface_hub

In [ ]:
# 셀 6 — 버전 확인
import torch
import transformers
import datasets
import peft
import trl
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("✅ 모든 라이브러리 확인 완료")

## ⚠️ 여기서 런타임 재시작!

상단 메뉴 → **런타임** → **런타임 재시작**  
재시작 후 아래 **셀 6.5 (CUDA 수정)** → **셀 7**부터 이어서 실행하세요.

---

In [ ]:
# 셀 6.5 — bitsandbytes CUDA 수정 (Day 025 해결법)
# 런타임 재시작 후, FastLanguageModel 로드 전에 반드시 실행!
import subprocess, ctypes

result = subprocess.run(["find", "/usr", "-name", "libnvJitLink.so.13"], capture_output=True, text=True)
paths = result.stdout.strip().split("\n")

if paths and paths[0]:
    ctypes.CDLL(paths[0], mode=ctypes.RTLD_GLOBAL)
    print(f"✅ CUDA 라이브러리 로드 완료: {paths[0]}")
else:
    print("⚠️ libnvJitLink.so.13을 찾지 못했습니다 — 그냥 다음 셀로 진행하세요")

In [ ]:
# 셀 7 — 모델 로드 (Day 026: Gemma→Llama 교체)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)
print("✅ 모델 로드 성공")
print(f"모델: Llama-3.2-1B-Instruct")

In [ ]:
# 셀 7 — FastLanguageModel 모델 로드
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2b-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("✅ 모델 로드 성공")

In [ ]:
# 셀 8 — LoRA 어댑터 설정
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ LoRA 어댑터 설정 완료")

In [ ]:
# 셀 9 — GPU 메모리 확인
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"전체 메모리: {max_memory} GB")
print(f"현재 사용 중: {start_gpu_memory} GB")
print(f"남은 메모리: {round(max_memory - start_gpu_memory, 3)} GB")

## 📄 SECTION 2 — Day 019: JSONL 포맷 분석

In [ ]:
# 셀 10 — JSONL 샘플 5개 작성 + 저장
import json

data = [
    {"instruction": "크라베오 회사를 소개해줘", "input": "", "output": "크라베오는 1인 기업 자동화를 연구하는 회사입니다. AI와 Python을 활용해 반복 업무를 줄이는 시스템을 만듭니다."},
    {"instruction": "파인튜닝이 뭔지 쉽게 설명해줘", "input": "", "output": "파인튜닝은 이미 만들어진 AI 모델을 내 데이터로 추가 학습시키는 과정입니다. AI가 내 스타일과 지식을 갖게 됩니다."},
    {"instruction": "Unsloth를 쓰는 이유가 뭐야?", "input": "", "output": "Unsloth는 파인튜닝 속도를 2~5배 빠르게 하고 메모리를 60% 절약합니다. 무료 코랩 환경에서 파인튜닝을 현실적으로 가능하게 해주는 도구입니다."},
    {"instruction": "LoRA가 뭐야?", "input": "", "output": "LoRA는 모델 전체를 학습시키지 않고 일부 층에만 작은 어댑터를 붙여서 학습하는 방법입니다. 메모리와 시간을 크게 절약할 수 있습니다."},
    {"instruction": "JSONL 포맷이 뭐야?", "input": "", "output": "JSONL은 JSON 데이터를 한 줄에 하나씩 나열한 파일 형식입니다. 파인튜닝 학습 데이터를 저장하는 데 표준으로 쓰입니다."},
]

with open("cravveo_sample.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ cravveo_sample.jsonl 생성 완료 — {len(data)}개")

In [ ]:
# 셀 11 — JSONL 파일 읽기 확인
with open("cravveo_sample.jsonl", "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"총 {len(lines)}줄\n")
for i, line in enumerate(lines):
    item = json.loads(line)
    print(f"[{i+1}] Q: {item['instruction']}")
    print(f"     A: {item['output'][:40]}...")
    print()

In [ ]:
# 셀 12 — HuggingFace datasets 라이브러리로 로드
from datasets import load_dataset

dataset = load_dataset("json", data_files="cravveo_sample.jsonl", split="train")
print(f"데이터셋 크기: {len(dataset)}개")
print(f"컬럼: {dataset.column_names}")
print()
for i in range(len(dataset)):
    print(f"[{i+1}] {dataset[i]['instruction']}")

## 📝 SECTION 3 — Day 020+021: 크라베오 데이터셋 제작

In [ ]:
# 셀 13 — 크라베오 정체성 데이터 10개 작성 + 저장
import json

cravveo_data = [
    {"instruction": "크라베오 컴퍼니가 뭐야?", "input": "", "output": "IT 프로그램 개발, 인터넷 쇼핑몰, 유튜브 채널 등을 혼자서 운영하면서 수익을 창출하는 1인 기업이야."},
    {"instruction": "왜 1인 기업을 선택했어?", "input": "", "output": "회사 생활을 20년 넘게 하면서 이리 치이고 저리 치이다 보니까 혼자 하는 게 낫겠다 싶었어. 그래서 시작했어."},
    {"instruction": "100일 AI 도전이 뭐야?", "input": "", "output": "지금 내가 하고 있는 것들을 홍보도 하고 기록도 남기고 싶어서 도전하고 있어."},
    {"instruction": "파인튜닝을 왜 배우고 있어?", "input": "", "output": "결국 개인의 지식만이 살아남는다는 걸 배우면서 느꼈어. 클라우드 AI 비용 문제도 있고, 나만의 AI를 만들면 파인튜닝으로 수익도 낼 수 있지 않을까 싶어서 배우고 있어."},
    {"instruction": "크라베오의 목표가 뭐야?", "input": "", "output": "1인 스타트업으로 살아남는 거야. 그게 지금 목표야."},
    {"instruction": "유튜브를 왜 시작했어?", "input": "", "output": "홍보랑 기록용이야. 내가 뭘 하고 있는지 남기고 싶었어."},
    {"instruction": "하루를 어떻게 보내?", "input": "", "output": "아침에 생산직 회사에 출근해서 일하고, 오후 6시쯤 퇴근하면 1인 스타트업 만들려고 공부하고 유튜브 영상 만들어. 저녁 12시쯤에 집에 가서 새벽 1~3시쯤에 자."},
    {"instruction": "AI를 처음 배울 때 가장 어려웠던 게 뭐야?", "input": "", "output": "시작하는 거야. 컴퓨터 켜고 도전해보는 게 제일 어려웠어. 이걸 왜 해야 하는지도 몰랐으니까."},
    {"instruction": "Build in Public이 뭐야?", "input": "", "output": "배우는 과정을 숨기지 않고 공개하면서 진행하는 방식이야. 완성된 결과만 보여주는 게 아니라 실패도 에러도 다 공개하는 거야. 내가 100일 도전을 유튜브에 올리는 것도 Build in Public이야."},
    {"instruction": "크라베오 AI가 완성되면 뭘 할 거야?", "input": "", "output": "당연히 수익을 창출해야지. 그게 목적이야."},
]

with open("cravveo_identity.jsonl", "w", encoding="utf-8") as f:
    for item in cravveo_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ cravveo_identity.jsonl 저장 완료 — {len(cravveo_data)}개")

In [ ]:
# 셀 14 — JSONL 형식 검증
print("--- JSONL 검증 ---\n")
errors = 0
with open("cravveo_identity.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        try:
            item = json.loads(line)
            print(f"[{i+1}] ✅ {item['instruction'][:30]}")
        except json.JSONDecodeError as e:
            print(f"[{i+1}] ❌ 형식 오류: {e}")
            errors += 1

print(f"\n총 오류: {errors}개" if errors else "\n✅ 모든 항목 형식 정상")

In [ ]:
# 셀 15 — 전체 데이터셋 합치기
# cravveo_briefing_dataset.jsonl 파일을 미리 업로드해두세요
# 코랩 왼쪽 파일 탭 → 업로드 버튼

source_files = [
    "cravveo_briefing_dataset.jsonl",
    "cravveo_identity.jsonl",
]

all_data = []
for fname in source_files:
    try:
        with open(fname, "r", encoding="utf-8") as f:
            lines = f.readlines()
            all_data.extend(lines)
            print(f"✅ {fname} — {len(lines)}개 로드")
    except FileNotFoundError:
        print(f"⚠️ {fname} 없음 — 건너뜀")

with open("cravveo_full_dataset.jsonl", "w", encoding="utf-8") as f:
    f.writelines(all_data)

print(f"\n총 {len(all_data)}개 → cravveo_full_dataset.jsonl 완성")

In [ ]:
# 셀 16 — 전체 데이터셋 샘플 확인
with open("cravveo_full_dataset.jsonl", "r", encoding="utf-8") as f:
    all_lines = f.readlines()

total = len(all_lines)
print(f"전체 데이터: {total}개\n")

print("=== 처음 3개 ===")
for line in all_lines[:3]:
    item = json.loads(line)
    print(f"Q: {item['instruction']}")
    print(f"A: {item['output'][:50]}...")
    print()

print("=== 마지막 3개 ===")
for line in all_lines[-3:]:
    item = json.loads(line)
    print(f"Q: {item['instruction']}")
    print(f"A: {item['output'][:50]}...")
    print()

## 🤗 SECTION 4 — Day 022: HuggingFace 로그인

In [ ]:
# 셀 17 — HuggingFace 로그인 (Colab Secrets 방식)
# 사전 준비: 코랩 왼쪽 🔑 아이콘 → + 새 비밀 추가
#   이름: HF_TOKEN
#   값: 허깅페이스 토큰 붙여넣기
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("✅ HuggingFace 로그인 완료")

In [ ]:
# 셀 18 — 로그인 확인
from huggingface_hub import whoami

user_info = whoami()
print(f"✅ 로그인 성공!")
print(f"사용자명: {user_info['name']}")

## 📥 SECTION 5 — Day 024: HuggingFace 데이터셋 연결

In [ ]:
# 셀 19 — HuggingFace에서 내 데이터셋 불러오기
from datasets import load_dataset

dataset = load_dataset("cravveo/cravveo-briefing-dataset", split="train")

print(f"✅ 데이터셋 로드 완료")
print(f"데이터 수: {len(dataset)}개")
print(f"컬럼: {dataset.column_names}")
print()
print("--- 첫 번째 데이터 미리보기 ---")
print("instruction:", dataset[0]['instruction'][:80])
print("output:", dataset[0]['output'][:80])

In [ ]:
# 셀 20 — 프롬프트 변환 (Day 026: Llama 3.2용 단순 포맷)
dataset_clean = dataset.filter(lambda x: len(x["output"]) <= 300)
print(f"원본: {len(dataset)}개 → 필터링 후: {len(dataset_clean)}개")

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    for instruction, output in zip(examples["instruction"], examples["output"]):
        text = f"질문: {instruction}\n답변: {output}{EOS_TOKEN}"
        texts.append(text)
    return {"text": texts}

dataset = dataset_clean.map(formatting_prompts_func, batched=True)

print("✅ 변환 완료")
print()
print("--- 변환 결과 확인 ---")
print(dataset[0]['text'][:300])

In [ ]:
# 셀 21 — SFTTrainer 설정 (Day 026: Llama 3.2 + 수정된 설정)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 2,
        num_train_epochs = 3,
        learning_rate = 2e-5,
        fp16 = True,
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
        seed = 42,
        save_strategy = "no",
    ),
)

print("✅ SFTTrainer 설정 완료")

In [ ]:
# 셀 21 — SFTTrainer 설정
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
        seed = 42,
    ),
)

print("✅ SFTTrainer 설정 완료")

In [ ]:
# 셀 22 — 학습 실행
print("🚀 학습 시작!")
trainer_stats = trainer.train()
print("✅ 학습 완료!")
print(f"총 학습 시간: {trainer_stats.metrics['train_runtime']:.1f}초")
print(f"최종 loss: {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
# 셀 23 — Day 025 vs Day 026 loss 비교
print("=" * 50)
print("📊 Day 025 vs Day 026 비교")
print("=" * 50)
print()
print("Day 025 (### 질문/답변 형식):")
print("  Step 1 loss:  25.5")
print("  Step 30 loss: ~15.4")
print()
print("Day 026 (Gemma 채팅 템플릿):")
print(f"  최종 loss: {trainer_stats.metrics['train_loss']:.4f}")
print()
if trainer_stats.metrics['train_loss'] < 5:
    print("✅ 프롬프트 템플릿 수정 성공! loss가 정상 범위입니다.")
else:
    print("⚠️ loss가 아직 높습니다. 데이터 확인이 필요합니다.")